# Human-in-the-Loop

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will learn how to pause graph execution with `interrupt(value)`, resume with `Command(resume=value)`, and implement approval and review/edit patterns.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Interrupt and Resume

`interrupt(value)` pauses graph execution and surfaces data. `Command(resume=value)` continues from the pause point.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]
    human_input: str

def ask_human(state: State) -> dict:
    response = interrupt({"question": "What is your favorite programming language?"})
    return {"human_input": response}

def respond(state: State) -> dict:
    return {"messages": [("ai", f"Great choice! {state['human_input']} is awesome.")]}

graph = StateGraph(State)
graph.add_node("ask_human", ask_human)
graph.add_node("respond", respond)
graph.add_edge(START, "ask_human")
graph.add_edge("ask_human", "respond")
graph.add_edge("respond", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "hitl-1"}}
result = app.invoke({"messages": []}, config=config)
print("Graph paused at interrupt.")

In [ ]:
state = app.get_state(config)
print(f"Next node: {state.next}")
print(f"Interrupt value: {state.tasks[0].interrupts[0].value}")

In [ ]:
result = app.invoke(Command(resume="Python"), config=config)
print(f"Response: {result['messages'][-1].content}")

## 4. Approval Pattern

The agent proposes an action and the human approves or rejects before proceeding.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

class ApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    approved: bool

def generate(state: ApprovalState) -> dict:
    response = llm.invoke([
        ("system", "Draft a short email based on the user request."),
        *state["messages"],
    ])
    return {"messages": [response]}

def human_review(state: ApprovalState) -> dict:
    last_msg = state["messages"][-1].content
    decision = interrupt({"draft": last_msg, "instruction": "Type 'approve' or 'reject'"})
    return {"approved": decision == "approve"}

def finalize(state: ApprovalState) -> dict:
    if state["approved"]:
        return {"messages": [("ai", "Email sent!")]}
    return {"messages": [("ai", "Email discarded.")]}

graph = StateGraph(ApprovalState)
graph.add_node("generate", generate)
graph.add_node("human_review", human_review)
graph.add_node("finalize", finalize)
graph.add_edge(START, "generate")
graph.add_edge("generate", "human_review")
graph.add_edge("human_review", "finalize")
graph.add_edge("finalize", END)

checkpointer2 = InMemorySaver()
app2 = graph.compile(checkpointer=checkpointer2)

In [ ]:
config2 = {"configurable": {"thread_id": "email-1"}}

result = app2.invoke(
    {"messages": [("human", "Write an email to cancel my gym membership")]},
    config=config2,
)
print("Draft generated. Waiting for approval...")

state = app2.get_state(config2)
print(f"\nDraft:\n{state.tasks[0].interrupts[0].value['draft']}")

In [ ]:
result = app2.invoke(Command(resume="approve"), config=config2)
print(f"Result: {result['messages'][-1].content}")

## 5. Review / Edit Pattern

The human can return edited content instead of a simple yes/no.

In [ ]:
class EditState(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str

def write_draft(state: EditState) -> dict:
    response = llm.invoke([
        ("system", "Write a short social media post based on the request."),
        *state["messages"],
    ])
    return {"draft": response.content}

def human_edit(state: EditState) -> dict:
    edited = interrupt({"draft": state["draft"], "instruction": "Edit or return as-is"})
    return {"draft": edited}

def publish(state: EditState) -> dict:
    return {"messages": [("ai", f"Published: {state['draft']}")]}

graph = StateGraph(EditState)
graph.add_node("write_draft", write_draft)
graph.add_node("human_edit", human_edit)
graph.add_node("publish", publish)
graph.add_edge(START, "write_draft")
graph.add_edge("write_draft", "human_edit")
graph.add_edge("human_edit", "publish")
graph.add_edge("publish", END)

checkpointer3 = InMemorySaver()
app3 = graph.compile(checkpointer=checkpointer3)

config3 = {"configurable": {"thread_id": "post-1"}}
app3.invoke({"messages": [("human", "Announce our new AI product")]}, config=config3)
print("Draft ready for editing...")

state = app3.get_state(config3)
print(f"Draft: {state.tasks[0].interrupts[0].value['draft']}")

In [ ]:
result = app3.invoke(
    Command(resume="Excited to announce our groundbreaking AI assistant! Try it today."),
    config=config3,
)
print(f"Result: {result['messages'][-1].content}")

## What You Learned

1. **`interrupt(value)`** pauses graph execution and surfaces data for human review
2. **`Command(resume=value)`** continues the graph from the exact interrupt point
3. A **checkpointer is required** — it saves state at the interrupt point
4. The **approval pattern** uses interrupt for yes/no decisions
5. The **review/edit pattern** lets humans modify output by returning edited content